# 07 — Assets & Lifestyle
**Source:** India Districts Census 2011 · 640 districts  
Covers household asset ownership (TV, mobile, bicycle, car, scooter, radio) as a proxy for economic well-being.

In [ ]:
import sqlite3, pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import warnings; warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid'); plt.rcParams['figure.dpi'] = 130

DB = '../data/processed/census_model.db'
conn = sqlite3.connect(DB)

df = pd.read_sql("""
    SELECT d.state_name, d.district_name,
           f.population, f.households,
           f.households_with_television,
           f.households_with_telephone_mobile_phone,
           f.households_with_bicycle,
           f.households_with_car_jeep_van,
           f.households_with_scooter_motorcycle_moped,
           f.households_with_radio_transistor,
           f.households_with_computer,
           f.households_with_tv_computer_laptop_telephone_mobile_phone_and_scooter_car
    FROM fact_census f
    JOIN dim_location d ON f.location_id = d.location_id
""", conn)
conn.close()

df = df[df['households'] > 0]

asset_map = {
    'households_with_television':         'TV',
    'households_with_telephone_mobile_phone': 'Mobile Phone',
    'households_with_bicycle':            'Bicycle',
    'households_with_car_jeep_van':       'Car/Jeep/Van',
    'households_with_scooter_motorcycle_moped': 'Scooter/Motorcycle',
    'households_with_radio_transistor':   'Radio/Transistor',
    'households_with_computer':           'Computer',
    'households_with_tv_computer_laptop_telephone_mobile_phone_and_scooter_car': 'All Major Assets',
}

for col, label in asset_map.items():
    df[label+'_pct'] = (df[col] / df['households'] * 100).round(2)

# Asset Wealth Index: avg of pct ownership of all major assets
pct_cols = [label+'_pct' for label in asset_map.values()]
df['asset_index'] = df[pct_cols].mean(axis=1).round(2)

state = df.groupby('state_name').agg(
    households=('households','sum'),
    **{col: (col,'sum') for col in asset_map.keys()}
).reset_index()

for col, label in asset_map.items():
    state[label+'_pct'] = (state[col] / state['households'] * 100).round(2)

print('Assets loaded. Districts:', len(df))

## 7.1 — National Asset Ownership (Bar Chart)

In [ ]:
national_assets = {label: df[col].sum() / df['households'].sum() * 100
                   for col, label in asset_map.items() if label != 'All Major Assets'}

fig, ax = plt.subplots(figsize=(10, 6))
sorted_assets = dict(sorted(national_assets.items(), key=lambda x: x[1], reverse=True))
colors = sns.color_palette('viridis', len(sorted_assets))
bars = ax.barh(list(sorted_assets.keys()), list(sorted_assets.values()), color=colors)

for bar, val in zip(bars, sorted_assets.values()):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', va='center', fontsize=10)

ax.set_xlabel('% of Households that Own')
ax.set_title('National Asset Ownership Rate — India 2011 Census', fontsize=13, fontweight='bold')
ax.set_xlim(0, 110)
plt.tight_layout(); plt.show()

## 7.2 — Asset Ownership by State (Heatmap %)

In [ ]:
heat_labels = [label+'_pct' for label in asset_map.values() if label != 'All Major Assets']
heat_display = [label for label in asset_map.values() if label != 'All Major Assets']

heat = state.set_index('state_name')[heat_labels].copy()
heat.columns = heat_display
heat = heat.sort_values('TV', ascending=False)

fig, ax = plt.subplots(figsize=(13, 12))
sns.heatmap(heat, annot=True, fmt='.1f', cmap='YlOrRd', linewidths=.4,
            cbar_kws={'label': '% of Households'}, ax=ax)
ax.set_title('Household Asset Ownership by State (%) — sorted by TV ownership',
             fontsize=12, fontweight='bold')
ax.set_xlabel('')
plt.tight_layout(); plt.show()

## 7.3 — Mobile Phone vs TV Ownership (District Scatter)

In [ ]:
fig = px.scatter(
    df, x='TV_pct', y='Mobile Phone_pct',
    color='state_name', hover_name='district_name',
    size='households', size_max=22,
    labels={'TV_pct': 'TV Ownership (%)', 'Mobile Phone_pct': 'Mobile Phone Ownership (%)'},
    title='TV Ownership vs Mobile Phone Ownership — District Level',
    trendline='ols', trendline_scope='overall', trendline_color_override='black'
)
fig.update_layout(showlegend=False, height=550)
fig.show()

## 7.4 — Car/Van vs Computer Ownership (Wealth Proxy)

In [ ]:
fig = px.scatter(
    df, x='Car/Jeep/Van_pct', y='Computer_pct',
    color='state_name', hover_name='district_name',
    labels={'Car/Jeep/Van_pct':'Car/Jeep/Van Ownership (%)','Computer_pct':'Computer Ownership (%)'},
    title='Car/Van vs Computer Ownership — District Level (Both = High Wealth Proxy)'
)
fig.update_layout(showlegend=False, height=520)
fig.show()

## 7.5 — Asset Wealth Index — Top & Bottom 10 Districts

In [ ]:
top10    = df.nlargest(10, 'asset_index')[['district_name','state_name','asset_index']]
bottom10 = df.nsmallest(10, 'asset_index')[['district_name','state_name','asset_index']]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.barh(top10['district_name'] + ', ' + top10['state_name'], top10['asset_index'],
         color=sns.color_palette('Greens_r', 10))
ax1.set_xlabel('Asset Index (avg % ownership)')
ax1.set_title('Top 10: Highest Asset Ownership', fontweight='bold')
ax1.invert_yaxis()

ax2.barh(bottom10['district_name'] + ', ' + bottom10['state_name'], bottom10['asset_index'],
         color=sns.color_palette('Reds_r', 10))
ax2.set_xlabel('Asset Index (avg % ownership)')
ax2.set_title('Bottom 10: Lowest Asset Ownership', fontweight='bold')
ax2.invert_yaxis()

plt.tight_layout(); plt.show()

## 7.6 — All-Asset Radial Chart by State (Spider/Radar)

In [ ]:
# Select top 6 states by population for radar chart
top_states = df.groupby('state_name')['households'].sum().nlargest(6).index.tolist()
radar_cols  = [label+'_pct' for label in ['TV','Mobile Phone','Bicycle','Car/Jeep/Van','Scooter/Motorcycle','Computer']]
radar_labels= ['TV','Mobile Phone','Bicycle','Car/Van','Scooter/Motorcycle','Computer']

fig = go.Figure()
for s in top_states:
    row = state[state['state_name'] == s][radar_cols].values[0].tolist()
    row.append(row[0])  # close the shape
    angles = radar_labels + [radar_labels[0]]
    fig.add_trace(go.Scatterpolar(r=row, theta=angles, fill='toself', name=s))

fig.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 100])),
    title='Asset Ownership Radar — Top 6 States by Household Count',
    height=550
)
fig.show()